In [1]:
#Data Preprocessing
import pandas as pd
import numpy as np

DGS2 = pd.read_csv('data/DGS2.csv')
DGS5 = pd.read_csv('data/DGS5.csv')
DGS10 = pd.read_csv('data/DGS10.csv')    

print(DGS2.shape, DGS5.shape, DGS10.shape) #Print the shapes of the dataframes to verify successful loading
print(DGS2.info(),DGS5.info(), DGS10.info()) #Print the info of the dataframe to check data types and non-null counts

merged = DGS2.merge(DGS5, on="observation_date", how="inner").merge(DGS10, on="observation_date", how="inner") #Merge the dataframes on observation_date
merged = merged.dropna(subset=["DGS2","DGS5","DGS10"]) #Drop rows with missing values in any of the yield columns

merged = merged.set_index("observation_date") #Set observation_date as index
merged.index = pd.to_datetime(merged.index, errors="coerce") #Convert index to datetime



(4394, 2) (4394, 2) (4394, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4394 entries, 0 to 4393
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  4394 non-null   object 
 1   DGS2              4214 non-null   float64
dtypes: float64(1), object(1)
memory usage: 68.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4394 entries, 0 to 4393
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  4394 non-null   object 
 1   DGS5              4214 non-null   float64
dtypes: float64(1), object(1)
memory usage: 68.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4394 entries, 0 to 4393
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   observation_date  4394 non-null   object 
 1   DGS10             4214 n

# Preprocessing summary

-Loaded raw FRED CSVs (DGS2.csv, DGS5.csv, DGS10.csv) and inspected shapes/dtypes. We saw 4394 entries of which 4214 were non-null

-Merged the three series on the common key observation_date using an inner join.

-Dropped rows with missing yields in any of DGS2, DGS5, DGS10 (missing days are typically U.S. market holidays or occasional gaps) leaving us with the 4214 entries from before

-Converted observation_date to proper datetime and set it as the index to enable clean time-series operations

In [4]:
merged

,DGS2,DGS5,DGS10
observation_date,,,
2009-01-02,0.88,1.72,2.46
2009-01-05,0.78,1.67,2.49
2009-01-06,0.80,1.68,2.51
2009-01-07,0.82,1.66,2.52
2009-01-08,0.83,1.60,2.47
...,...,...,...
2025-10-30,3.61,3.72,4.11
2025-10-31,3.60,3.71,4.11
2025-11-03,3.60,3.72,4.13


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import kaleido

# Assuming your dataframe is called "merged"
# Columns like ['DGS2', 'DGS5', 'DGS10', ...], index = datetime

# 1. Resample to weekly (optional for smoother plotting)
weekly = merged.resample('W-FRI').mean().dropna()

# 2. Extract maturities (e.g. "DGS2" -> 2)
maturities = np.array([2, 5, 10])

# 3. Build meshgrid for plotting
X = np.arange(len(weekly))  # time index
Y = maturities
X_mesh, Y_mesh = np.meshgrid(X, Y)
Z = weekly.T.values  # shape (n_maturities, n_weeks)

# 4. Create interactive surface plot
fig = go.Figure(data=[
    go.Surface(
        x=X_mesh,
        y=Y_mesh,
        z=Z,
        colorscale='Viridis',
        colorbar=dict(
            title='Yield (%)',
            thickness=10,   # make it narrower
            len=0.7         # shorten it vertically a bit
        )
    )
])

# 5. Update layout
fig.update_layout(
    title='Weekly U.S. Treasury Yield Curves (2009–Today)',
    scene=dict(
        xaxis_title='Time Index (weekly)',
        yaxis_title='Maturity (years)',
        zaxis_title='Yield (%)',
        camera=dict(eye=dict(x=1.5, y=-1.3, z=0.8))
    ),
    autosize=True,
    width=900,
    height=600,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()


FileNotFoundError: [Errno 2] No such file or directory: 'figures/yield_surface.png'